# ChurnIQ — Telecom Customer Churn & Revenue Risk Analyzer

## Phase 7 — Business Insights & Churn Driver Analysis

### Objective

The predictive model identifies customers at risk of churn.

The objective of this phase is to identify:

- Key churn drivers
- Behavioral indicators of churn
- Revenue-related churn signals
- Customer lifecycle effects
- Business actions that may reduce churn

---

### Champion Model

XGBoost

Threshold: 0.40

Cross-Validated ROC-AUC: 0.680

Recall: 82.66%

F1 Score: 0.495

In [1]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Step 1: Load Final Modeling Dataset

Load the feature-engineered dataset and recreate the final leakage-safe feature set used by the champion XGBoost model.

In [2]:
# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv(
    "../data/processed/feature_engineered.csv"
)

print(df.shape)

(51047, 67)


C:\Users\Akshit\AppData\Local\Temp\ipykernel_6896\1420387941.py:5: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [3]:
# ==========================================
# TARGET ENCODING
# ==========================================

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [5]:
# ==========================================
# FINAL FEATURE SELECTION
# ==========================================

columns_to_drop = [
    "CustomerID",
    "ServiceArea",
    "MarketCode",
    "AreaCode",
    "CreditRating",
    "RetentionCalls",
    "RetentionOffersAccepted",
    "MadeCallToRetentionTeam"
]

model_df = df.drop(
    columns=columns_to_drop
)

print(model_df.shape)

(51047, 59)


In [6]:
# ==========================================
# FEATURES & TARGET
# ==========================================

X = model_df.drop(columns=["Churn"])
y = model_df["Churn"]

print(X.shape)
print(y.shape)

(51047, 58)
(51047,)


In [7]:
# ==========================================
# FEATURE TYPES
# ==========================================

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))

Numerical: 38
Categorical: 20


In [8]:
# ==========================================
# CLASS IMBALANCE
# ==========================================

negative_class = (y == 0).sum()
positive_class = (y == 1).sum()

scale_pos_weight = (
    negative_class / positive_class
)

print(round(scale_pos_weight, 2))

2.47


In [10]:
# ==========================================
# PREPROCESSOR
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [11]:
# ==========================================
# TRAIN CHAMPION MODEL
# ==========================================

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

xgb_pipeline.fit(X, y)

print("Champion model trained.")

Champion model trained.


In [12]:
# ==========================================
# FEATURE NAMES
# ==========================================

feature_names = xgb_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

print(len(feature_names))

144


## Step 2: Extract Feature Importance

XGBoost assigns an importance score to each feature based on its contribution to model decisions.

The objective is to identify:

- Top churn drivers
- Customer behavior signals
- Revenue-related risk indicators
- Actionable business insights

In [13]:
# ==========================================
# EXTRACT XGBOOST MODEL
# ==========================================

xgb_model = xgb_pipeline.named_steps["model"]

print(type(xgb_model))

<class 'xgboost.sklearn.XGBClassifier'>


In [14]:
# ==========================================
# FEATURE IMPORTANCE
# ==========================================

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df.head(20)

,Feature,Importance
47,categorical__CustomerLifecycleStage_New,0.038392
140,remainder__HandsetPrice_Missing,0.038207
132,remainder__CurrentEquipmentDays,0.023515
2,categorical__HandsetRefurbished_No,0.021458
127,remainder__MonthsInService,0.019683
19,categorical__NonUSTravel_Yes,0.014719
5,categorical__HandsetWebCapable_Yes,0.014166
4,categorical__HandsetWebCapable_No,0.014063
131,remainder__HandsetModels,0.012105
14,categorical__RespondsToMailOffers_No,0.011926


In [15]:
# ==========================================
# TOP 20 FEATURES
# ==========================================

top20_features = importance_df.head(20)

top20_features

,Feature,Importance
47,categorical__CustomerLifecycleStage_New,0.038392
140,remainder__HandsetPrice_Missing,0.038207
132,remainder__CurrentEquipmentDays,0.023515
2,categorical__HandsetRefurbished_No,0.021458
127,remainder__MonthsInService,0.019683
19,categorical__NonUSTravel_Yes,0.014719
5,categorical__HandsetWebCapable_Yes,0.014166
4,categorical__HandsetWebCapable_No,0.014063
131,remainder__HandsetModels,0.012105
14,categorical__RespondsToMailOffers_No,0.011926


In [16]:
# ==========================================
# IMPORTANCE COVERAGE
# ==========================================

top20_features["Importance"].sum()

np.float32(0.31351715)

In [17]:
top20_features.reset_index(drop=True)

,Feature,Importance
0,categorical__CustomerLifecycleStage_New,0.038392
1,remainder__HandsetPrice_Missing,0.038207
2,remainder__CurrentEquipmentDays,0.023515
3,categorical__HandsetRefurbished_No,0.021458
4,remainder__MonthsInService,0.019683
5,categorical__NonUSTravel_Yes,0.014719
6,categorical__HandsetWebCapable_Yes,0.014166
7,categorical__HandsetWebCapable_No,0.014063
8,remainder__HandsetModels,0.012105
9,categorical__RespondsToMailOffers_No,0.011926


In [18]:
importance_df.head(20)

,Feature,Importance
47,categorical__CustomerLifecycleStage_New,0.038392
140,remainder__HandsetPrice_Missing,0.038207
132,remainder__CurrentEquipmentDays,0.023515
2,categorical__HandsetRefurbished_No,0.021458
127,remainder__MonthsInService,0.019683
19,categorical__NonUSTravel_Yes,0.014719
5,categorical__HandsetWebCapable_Yes,0.014166
4,categorical__HandsetWebCapable_No,0.014063
131,remainder__HandsetModels,0.012105
14,categorical__RespondsToMailOffers_No,0.011926
